## Load all required libraries

In [11]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

## Load required images

In [6]:
def readImages(folder, num_images):
    arr_images = []
    for i in range(num_images):
        arr_images.append(cv2.imread(f'{folder}{i}.jpg'))
    return np.array(arr_images, dtype=np.float32)

folder = '../images/'
im = readImages(folder, 2)

## Step 1: Project the images onto a plane using homography transformation


### Setup

In [7]:
# You cannot use built-in functions to estimate the homography matrix,
# but you can use built-in functions to perform matrix operations, such as SVD decomposition, etc.

def estimate_homography_dlt(src_pts, dst_pts):
    """
    Estimate the homography matrix using Direct Linear Transformation (DLT) algorithm.
    Args:
        src_pts: np.ndarray of shape (n, 2) representing source points (x_i, y_i)
        dst_pts: np.ndarray of shape (n, 2) representing destination points (x'_i, y'_i)
    Returns:
        H: np.ndarray of shape (3, 3) Estimated homography matrix
    """

    n = src_pts.shape[0]
    assert dst_pts.shape[0] == n, "Number of source and destination points must be the same."
    assert n >= 4, "At least 4 point correspondences are required to estimate the homography matrix."

    A = []

    # Step 1: Construct the 2x9 matrix A_i for a pair of corresponding points
    def construct_A_i(src_pt, dst_pt):
        """
        Args:
            src_pt: np.ndarray of shape (2,) representing (x_i, y_i) coordinates of the source point
            dst_pt: np.ndarray of shape (2,) representing (x'_i, y'_i) coordinates of the destination point
        Returns:
            A_i: np.ndarray of shape (2, 9) representing the contribution of the i-th point correspondence to the matrix A
        """
        
        x, y = src_pt
        xx, yy = dst_pt
        
        # TODO: Your code here
        # A_i = ...  # (2, 9)
        
        A_i = np.array(
            [[-x, -y, -1, 0, 0, 0, x*xx, y*xx, xx],
            [0, 0, 0, -x, -y, -1, x*yy, y*yy, yy]])

        return A_i

    # Step 2: Stack the A_i matrices returned from the construct_A_i function to form a 2n x 9 matrix A
    # TODO: Your code here
    # A = []
    # for ... in range(n): ...
    # A = np.concatenate(...)
    
    A_temp = []
    
    for i in range(n):
        A_temp.append(construct_A_i(src_pts[i], dst_pts[i]))
    
    A = np.concatenate(A_temp)
    
    # print("A is now")
    # print(A)

    # Step 3: Perform Singular Value Decomposition (SVD) on the matrix A
    # TODO: Your code here
    # ... = np.linalg.svd(A)
    U, S, Vh = np.linalg.svd(A)

    # Step 4: The homography matrix H is the last column of V (or the last row of V^T) reshaped to 3x3
    # TODO: Your code here
    # h = .... # The last column of V
    # H = ...  # Reshape h to 3x3
    h = Vh[-1,:]
    H = h.reshape(3,3)

    # Step 5: Normalize the homography matrix H so that H[2, 2] = 1
    # TODO: Your code here
    # H = ...
    H = H / H[2,2]

    return H

In [8]:
def transform_points(points, H):
    """Apply homography transform to a set of points.
    You cannot use cv2.perspectiveTransform for this part. You need to implement the transformation yourself.

    Args:
        points: np.ndarray of shape (n, 2) representing n points in 2D space.
        H: np.ndarray of shape (3, 3) representing the homography matrix.

    Returns:
        np.ndarray of shape (n, 2) representing the transformed points.
    """
    # Step 1: Convert points to homogeneous coordinates -> make it to  (n, 3) where the last column is 1
    # TODO: your code here
    # points_hm = ...  # (n, 3)
    ones = np.ones(points.shape[0], dtype=np.float32)
    points_hm = np.column_stack((points, ones))
    # print(points_hm)

    # Step 2: Apply the homography transformation pt_hm' = H @ pt_hm
    # to process all n points together, you would need to be careful about the dimensions of the matrices.
    # You can choose either of the following options:
    # option 1: (n, 3) <- ((3, 3) @  (n, 3).T).T
    # option 2: (n, 3) <- (n, 3) @ (3, 3).T
    # TODO: your code here
    # transformed_points_hm = ...  # (n, 3)
    hm_transformation = np.shape(points_hm)
    hm_transformation = np.transpose(H @ np.transpose(points_hm))
    
    # print(hm_transformation)

    # Step 3: dividing the first two columns by the third column and discard the third column
    # to convert back to Cartesian coordinates (n, 2)
    # transformed_points = ... # (n, 2)
    transformed_points = hm_transformation[:, 0:2] / hm_transformation[:, [2]]
    # print(hm_transformation[:, 0:2])
    # print(hm_transformation[:, [2]])
    # print(transformed_points)
    # print("Finishsed")

    return transformed_points


In [9]:
def check_inliers(src_pts, tgt_pts, H, threshold=5.0):
    """
    Check which points are inliers given a homography matrix H and a set of point correspondences (src_pts, tgt_pts).
    Args:
        src_pts: np.ndarray of shape (n, 2) representing source points (x_i, y_i)
        tgt_pts: np.ndarray of shape (n, 2) representing destination points (x'_i, y'_i)
        H: np.ndarray of shape (3, 3) representing the homography matrix
        threshold: float, the distance threshold to determine if a point is an inlier
    Returns:
        inliers: np.ndarray of shape (n, ) boolean array where True indicates an inlier and False indicates an outlier
    """

    # step 1: apply the homography transformation to the source points to get the transformed points
    # TODO: your code here
    # transformed_pts = transform_points(...)  # (n, 2)
    transform_pts = transform_points(src_pts, H)

    # step 2: compute the distances between the transformed points and the target points
    # please compute the Euclidean distance (L2 distance) between each pair of transformed point and target point
    # you cannot use built-in functions, such as np.linalg.norm, to compute the distance, you need to implement it yourself using basic numpy operations.
    # distances = ...  # (n,)
    
    # print(transform_pts)
    # print(tgt_pts)
    
    diff = transform_pts - tgt_pts
    distances = np.sqrt(np.sum(diff**2, axis=1))
    # print(distances)

    # step 3: determine which points are inliers based on the distance and the threshold
    inliers = distances < threshold

    return inliers

In [10]:
def estimate_homography_ransac(src_pts, dst_pts, num_iters=1000, threshold=5.0):
    
    n = src_pts.shape[0]
    assert n >= 4, "At least 4 point correspondences are required to estimate the homography matrix."

    max_inliers = np.all(np.zeros(n, dtype=bool))  # (n,)

    for _ in range(num_iters):
        sample_indices = np.random.choice(len(src_pts), (4, ), replace=False)
        sample_src_pts = src_pts[sample_indices]
        sample_dst_pts = dst_pts[sample_indices]
        H = estimate_homography_dlt(sample_src_pts, sample_dst_pts)
        
        inliers = check_inliers(src_pts=src_pts, tgt_pts=dst_pts, H=H, threshold=threshold)
        if np.sum(inliers) > np.sum(max_inliers):
            max_inliers = inliers
            
    all_inlier_src_pts = src_pts[max_inliers]
    all_inlier_dst_pts = dst_pts[max_inliers]
    best_H = estimate_homography_dlt(all_inlier_src_pts, all_inlier_dst_pts)

    return best_H, max_inliers


### Map Image to Plane

In [ ]:
def project_and_crop_images():
    pass

project_and_crop_images()